# Imports

In [ ]:
import datetime
import ipywidgets as widgets
import re
import time

from collections import OrderedDict
from ipywidgets import interact
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.signal as signal
import tomli_w
import tomli

# Functions

In [ ]:
def plot_signal(
    data: pd.DataFrame, 
    sample_interval: float = 2
) -> tuple[plt.figure, plt.axis]:
    if data is None: return None
    
    fig, ax = plt.subplots(figsize=(15,8))
    ax.plot(
        np.arange(0, data.shape[0] * sample_interval, sample_interval), 
        data
    )

    ax.grid(visible=True)
    ax.tick_params(axis='both', labelsize=14)
    
    ax.set_xlabel("time (ns)", fontsize=18)
    ax.set_ylabel("amplitude ($V$)", fontsize=18)
    
    return fig, ax


def plot_bases(
    signal: pd.Series, 
    peak_idx: int,
    peak_height: float,
    left_idx: int, 
    right_idx: int,
    sample_interval: float = 2
) -> tuple[plt.figure, plt.axis]:
    
    if signal is None:
        return None
    
    fig, ax = plot_signal(signal, sample_interval)

    # Plot Peak
    ax.plot(
        peak_idx * sample_interval, 
        peak_height, 
        'rx'
    )
    
    # Left Base
    ax.plot(
        left_idx * sample_interval,
        signal[left_idx] - 0.02,
        'g^'
    )
    
    # Right Base
    ax.plot(
        right_idx * sample_interval,
        signal[right_idx] - 0.02,
        'g^'
    )
    
    ax.set_title(f"Plot of Signal {signal.name}")

In [ ]:
def generate_psd(df: pd.DataFrame, left_idxs, tail_offset: int=1) -> pd.DataFrame:
    def process(series: pd.Series) -> dict:
        left_idx = left_idxs[series.name][0]
        q_total = series[left_idx:].sum()
        q_tail = series[left_idx + tail_offset:].sum()

        res = {
            "q_total": q_total,
            "q_tail": q_tail,
            "quotient": q_tail/q_total
        }

        return res
    
    return pd.DataFrame({signal_id: process(df[signal_id]) for signal_id in df.columns})

# Pre-Processing

In [ ]:
EXP_ROOT = Path("../sample_datasets/20220824_CERC_background/processed_data/cleaned_buffers/")
PARQ_PATH = EXP_ROOT / "20220824-0003_clean.parquet"

In [ ]:
df = pd.read_parquet(PARQ_PATH)
df.columns = df.columns.astype("int16")
df = df.T

## Normalization

In [ ]:
# Normalize
df_norm = (df - df.min()) / (df.max() - df.min())

In [ ]:
plot_signal(df_norm)
plt.show()

## Smoothing

In [ ]:
signal.savgol_filter()

In [ ]:
df_processed = df_norm

# Pulse Shape Discrimination

## Peak Finding
Since we already found peaks in the `data_cleaning.ipynb` notebook, we can just reuse the settings to determine the same values! In a more complete notebook, we can just reuse these values instead of needing to recompute any information using `scipy.signal.find_peaks()`

In [ ]:
SETTINGS_PATH = EXP_ROOT / "settings.toml"

with open(SETTINGS_PATH, "rb") as f:
    exp_info = tomli.load(f)

multipeak_filter_settings = exp_info["multipeak_filter_settings"]
height = multipeak_filter_settings["height"]
prominence = multipeak_filter_settings["prominence"]

output = df_processed.apply(lambda x: signal.find_peaks(x, height=height, prominence=prominence))
peak_idx, props = output.iloc[0,:], output.iloc[1,:]

In [ ]:
def get_left_bases(series: pd.Series, peak_idx: int, peak_offset: int=10):
    return series[peak_idx - peak_offset: peak_idx].idxmin()

peak_offset = 20
left_bases = df_processed.apply(lambda x: get_left_bases(x, peak_idx[x.name][0], peak_offset))

### Visualize Peak Finding

In [ ]:
random_sample = df_processed.sample(10, axis=1, random_state=42)
sample_ids = random_sample.columns

interact(
    lambda signal_id: plot_bases(
        df_processed.get(signal_id), 
        peak_idx[signal_id],
        props[signal_id]["peak_heights"],
        # props[signal_id]["left_bases"],
        left_bases[signal_id],
        props[signal_id]["right_bases"]
    ), 
    signal_id=sample_ids
)

plt.show()

## Integration

**ASSUMPTION:** `left_base` is where the integration will begin, regardless of how far from the immediate base of the peak.

In [ ]:
start_time = time.perf_counter()

tail_offset = 0
left_idxs = {signal_id: props[signal_id]["left_bases"] for signal_id in props.keys()}
psd_report = generate_psd(df_processed, left_idxs, tail_offset)

print(
    f"Generated PSD report in [\x1b[1;32m{(time.perf_counter() - start_time)*1000:.2f} ms\x1b[0m]."
)

In [ ]:
psd_report.head()

## Visualizations

### <span style="color:#FF9900">Figure of Merit</span>

## Report